In [ ]:
"""
DEAP-FAKED Implementation with DBpedia (Alternative to Wikidata)
Since Wikidata5M is not accessible, using DBpedia as KG source
Following DEAP-FAKED paper methodology but with DBpedia
"""

'\nDEAP-FAKED Implementation with DBpedia (Alternative to Wikidata)\nSince Wikidata5M is not accessible, using DBpedia as KG source\nFollowing DEAP-FAKED paper methodology but with DBpedia\n'

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import pickle
import json
import os
import requests
import time
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
import random
import re
from collections import Counter
import spacy
from typing import Dict, List, Optional, Tuple, Any
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

class DeapConfig:
    # Model parameters (from DEAP-FAKED paper)
    vocab_size = 10000
    embedding_dim = 300
    hidden_dim = 256
    kg_embed_dim = 300  # DBpedia embeddings typically 300D
    max_seq_length = 256

    # Training parameters (from DEAP-FAKED paper)
    learning_rate = 0.001
    batch_size = 32
    num_epochs = 100
    early_stop_patience = 2
    dropout = 0.3

    # File paths
    train_path = "train_deap.csv"
    val_path = "val_deap.csv"
    test_path = "test_deap.csv"
    entity_stats_path = "entity_linking_stats_deap.csv"

    # Output files
    dbpedia_embeddings_path = "dbpedia_embeddings.pkl"
    vocab_path = "vocab.pkl"
    results_dir = "deap_faked_dbpedia_results"

    # DBpedia SPARQL endpoint
    dbpedia_sparql = "https://dbpedia.org/sparql"
    dbpedia_lookup = "http://lookup.dbpedia.org/api/search"

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = DeapConfig()

In [ ]:
# ============================================================================
# STEP 1: DBpedia Entity Linker
# ============================================================================

class DBpediaLinker:
    """Link entities to DBpedia and create embeddings"""

    @staticmethod
    def query_dbpedia_sparql(query: str) -> List[Dict]:
        """Query DBpedia SPARQL endpoint"""
        try:
            headers = {'Accept': 'application/json'}
            params = {'query': query, 'format': 'json'}
            response = requests.get(config.dbpedia_sparql, params=params,
                                  headers=headers, timeout=30)

            if response.status_code == 200:
                results = response.json()
                return results.get('results', {}).get('bindings', [])
        except Exception as e:
            print(f"SPARQL query error: {e}")
        return []

    @staticmethod
    def get_dbpedia_uri(entity_name: str) -> Optional[str]:
        """Get DBpedia URI for an entity name"""
        try:
            # Clean entity name
            entity_name = entity_name.replace('"', '').replace("'", "").strip()

            # Try SPARQL query first
            query = f"""
            SELECT DISTINCT ?entity WHERE {{
              ?entity rdfs:label "{entity_name}"@en .
              FILTER (STRSTARTS(STR(?entity), "http://dbpedia.org/resource/"))
            }} LIMIT 1
            """

            results = DBpediaLinker.query_dbpedia_sparql(query)
            if results:
                return results[0]['entity']['value']

            # Try API lookup as fallback
            params = {
                'query': entity_name,
                'format': 'json',
                'maxResults': 1
            }
            response = requests.get(config.dbpedia_lookup, params=params, timeout=30)

            if response.status_code == 200:
                results = response.json().get('docs', [])
                if results:
                    return results[0].get('resource', [None])[0]

        except Exception as e:
            print(f"Error getting DBpedia URI for {entity_name}: {str(e)[:50]}")

        return None

    @staticmethod
    def extract_dbpedia_id(uri: str) -> str:
        """Extract DBpedia ID from URI"""
        if not uri:
            return ""
        # Example: http://dbpedia.org/resource/Barack_Obama -> Barack_Obama
        return uri.split('/')[-1]

    @staticmethod
    def create_dbpedia_embeddings_from_entities(entity_names: List[str]) -> Dict[str, np.ndarray]:
        """Create embeddings for DBpedia entities using word embeddings"""
        print("\nCreating DBpedia entity embeddings...")

        if os.path.exists(config.dbpedia_embeddings_path):
            print("Loading existing DBpedia embeddings...")
            with open(config.dbpedia_embeddings_path, 'rb') as f:
                return pickle.load(f)

        # For simplicity, create embeddings based on entity names
        # In a real implementation, you would use pre-trained DBpedia embeddings
        # or extract embeddings from entity descriptions

        entity_embeddings = {}

        # Try to download pre-trained word embeddings for entity name components
        try:
            # Use GloVe embeddings for word components
            print("Downloading GloVe embeddings for entity representations...")
            import zipfile

            glove_url = "http://nlp.stanford.edu/data/glove.6B.300d.zip"
            glove_zip = "glove.6B.300d.zip"
            glove_file = "glove.6B.300d.txt"

            if not os.path.exists(glove_file):
                print("Downloading GloVe embeddings...")
                response = requests.get(glove_url, stream=True)
                with open(glove_zip, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)

                # Extract
                with zipfile.ZipFile(glove_zip, 'r') as zip_ref:
                    zip_ref.extractall('.')

                # Clean up
                if os.path.exists(glove_zip):
                    os.remove(glove_zip)

            # Load GloVe embeddings
            print("Loading GloVe embeddings...")
            glove_embeddings = {}
            with open(glove_file, 'r', encoding='utf-8') as f:
                for line in tqdm(f, desc="Loading GloVe"):
                    values = line.split()
                    word = values[0]
                    vector = np.asarray(values[1:], dtype='float32')
                    glove_embeddings[word] = vector

            # Create entity embeddings by averaging word embeddings
            for entity_name in tqdm(entity_names, desc="Creating entity embeddings"):
                # Clean entity name
                clean_name = re.sub(r'[^\w\s]', '', entity_name)
                words = clean_name.lower().split()

                # Get embeddings for each word
                word_vectors = []
                for word in words:
                    if word in glove_embeddings:
                        word_vectors.append(glove_embeddings[word])
                    elif len(word) > 3:  # Try subwords for longer words
                        for i in range(len(word) - 3):
                            subword = word[i:i+4]
                            if subword in glove_embeddings:
                                word_vectors.append(glove_embeddings[subword])

                if word_vectors:
                    # Average word vectors
                    entity_embedding = np.mean(word_vectors, axis=0)
                    # Normalize
                    norm = np.linalg.norm(entity_embedding)
                    if norm > 0:
                        entity_embedding = entity_embedding / norm
                    entity_embeddings[entity_name] = entity_embedding
                else:
                    # Create random embedding as fallback
                    rng = np.random.RandomState(hash(entity_name) % 1000000)
                    embedding = rng.randn(config.kg_embed_dim).astype(np.float32)
                    norm = np.linalg.norm(embedding)
                    if norm > 0:
                        embedding = embedding / norm
                    entity_embeddings[entity_name] = embedding

            print(f"Created {len(entity_embeddings):,} entity embeddings using GloVe")

        except Exception as e:
            print(f"Error using GloVe: {e}")
            print("Creating simple deterministic embeddings...")

            # Fallback: create deterministic embeddings
            for entity_name in entity_names:
                # Create deterministic embedding based on entity name hash
                import hashlib
                name_hash = int(hashlib.md5(entity_name.encode()).hexdigest(), 16) % 1000000
                rng = np.random.RandomState(name_hash)
                embedding = rng.randn(config.kg_embed_dim).astype(np.float32)
                # Normalize
                norm = np.linalg.norm(embedding)
                if norm > 0:
                    embedding = embedding / norm
                entity_embeddings[entity_name] = embedding

        # Save embeddings
        with open(config.dbpedia_embeddings_path, 'wb') as f:
            pickle.dump(entity_embeddings, f)

        print(f"Saved {len(entity_embeddings):,} DBpedia entity embeddings")
        return entity_embeddings

In [ ]:
# ============================================================================
# STEP 2: Data Loader
# ============================================================================

class DataLoaderManager:
    """Load and manage your preprocessed data"""

    @staticmethod
    def setup_directories():
        """Create necessary directories"""
        os.makedirs(config.results_dir, exist_ok=True)
        print(f"Results will be saved to: {config.results_dir}")

    @staticmethod
    def load_preprocessed_data():
        """Load your preprocessed datasets"""
        print("Loading preprocessed data...")

        # Check if files exist
        required_files = [
            config.train_path,
            config.val_path,
            config.test_path,
            config.entity_stats_path
        ]

        missing_files = []
        for file in required_files:
            if not os.path.exists(file):
                missing_files.append(file)

        if missing_files:
            print(f"ERROR: Missing files: {missing_files}")
            print("Please ensure these files are uploaded:")
            print("  - train_deap.csv")
            print("  - val_deap.csv")
            print("  - test_deap.csv")
            print("  - entity_linking_stats_deap.csv")
            return None, None, None

        # Load datasets
        train_df = pd.read_csv(config.train_path)
        val_df = pd.read_csv(config.val_path)
        test_df = pd.read_csv(config.test_path)

        # Verify columns
        for df, name in [(train_df, 'train'), (val_df, 'val'), (test_df, 'test')]:
            if 'title' not in df.columns or 'label' not in df.columns:
                print(f"ERROR: {name} dataset missing required columns")
                return None, None, None

        print(f"Train samples: {len(train_df):,}")
        print(f"Validation samples: {len(val_df):,}")
        print(f"Test samples: {len(test_df):,}")

        # Print class distribution
        print("\nClass Distribution:")
        for df, name in [(train_df, 'Train'), (val_df, 'Validation'), (test_df, 'Test')]:
            total = len(df)
            real = len(df[df['label'] == 0])
            fake = len(df[df['label'] == 1])
            print(f"  {name}: Real={real:,} ({real/total*100:.1f}%), Fake={fake:,} ({fake/total*100:.1f}%)")

        return train_df, val_df, test_df

    @staticmethod
    def extract_entity_names():
        """Extract entity names from preprocessing output"""
        print("\nExtracting entity names from preprocessing...")

        if not os.path.exists(config.entity_stats_path):
            print(f"ERROR: {config.entity_stats_path} not found")
            return []

        df = pd.read_csv(config.entity_stats_path)
        entity_names = set()

        for idx, row in tqdm(df.iterrows(), total=min(len(df), 10000), desc="Processing entity stats"):
            linked_entities = row['linked_entities']

            if pd.isna(linked_entities) or linked_entities == '[]':
                continue

            # Parse the string representation
            try:
                # Clean the string
                linked_str = str(linked_entities)

                # Try to parse as Python list
                import ast
                entities_list = ast.literal_eval(linked_str)

                for entity_tuple in entities_list:
                    if isinstance(entity_tuple, tuple) and len(entity_tuple) >= 2:
                        entity_name = str(entity_tuple[0]).strip()
                        entity_names.add(entity_name)
            except:
                # Alternative parsing
                try:
                    entities_str = str(linked_entities).strip('[]')
                    if entities_str:
                        items = entities_str.split('), (')
                        for item in items:
                            item = item.strip('()')
                            parts = item.split(', ')
                            if len(parts) >= 2:
                                entity_name = parts[0].strip("'\"")
                                entity_names.add(entity_name)
                except:
                    continue

        entity_names = list(entity_names)
        print(f"Extracted {len(entity_names):,} unique entity names")
        return entity_names

In [ ]:
# ============================================================================
# STEP 3: Entity Processor with spaCy
# ============================================================================

class EntityProcessor:
    """Process entities using spaCy NER"""

    def __init__(self):
        print("\nInitializing spaCy NER model...")
        try:
            self.nlp = spacy.load("en_core_web_sm")
            print("spaCy model loaded")
        except:
            print("Installing spaCy model...")
            import subprocess
            subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm", "-q"])
            self.nlp = spacy.load("en_core_web_sm")

        # Entity types to extract
        self.target_entity_types = {'PERSON', 'ORG', 'GPE', 'LOC'}

    def extract_entities(self, text: str) -> List[str]:
        """Extract entities from text using spaCy NER"""
        if not text or not isinstance(text, str):
            return []

        try:
            doc = self.nlp(text)
            entities = []
            for ent in doc.ents:
                if ent.label_ in self.target_entity_types:
                    entity_text = ent.text.strip()
                    if entity_text and len(entity_text) > 1:
                        entities.append(entity_text)
            return list(set(entities))
        except:
            return []

In [ ]:
# ============================================================================
# STEP 4: Dataset Class
# ============================================================================

class DeapFakedDataset(Dataset):
    """Dataset class for DEAP-FAKED model with DBpedia"""

    def __init__(self, df: pd.DataFrame, vocab: Dict[str, int],
                 entity_processor: EntityProcessor,
                 entity_embeddings: Dict[str, np.ndarray]):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.max_len = config.max_seq_length
        self.entity_processor = entity_processor
        self.entity_embeddings = entity_embeddings

        # Caches for performance
        self.sequence_cache = {}
        self.entity_embedding_cache = {}

    def __len__(self) -> int:
        return len(self.df)

    def text_to_sequence(self, text: str) -> List[int]:
        """Convert text to sequence of word indices"""
        cache_key = hash(text)

        if cache_key in self.sequence_cache:
            return self.sequence_cache[cache_key]

        text = str(text).lower().strip()
        words = text.split()

        sequence = []
        for word in words[:self.max_len]:
            if word in self.vocab:
                sequence.append(self.vocab[word])
            else:
                sequence.append(1)  # UNK token

        if len(sequence) < self.max_len:
            sequence = sequence + [0] * (self.max_len - len(sequence))

        result = sequence[:self.max_len]
        self.sequence_cache[cache_key] = result
        return result

    def get_entity_embedding(self, text: str) -> np.ndarray:
        """Get aggregated entity embedding for text"""
        if text in self.entity_embedding_cache:
            return self.entity_embedding_cache[text]

        # Extract entities
        entities = self.entity_processor.extract_entities(text)

        # Get embeddings for each entity
        entity_embeddings_list = []
        for entity in entities[:5]:  # Limit to 5 entities as in paper
            if entity in self.entity_embeddings:
                embedding = self.entity_embeddings[entity]
                entity_embeddings_list.append(embedding)
            else:
                # Try to find similar entity
                for emb_entity in self.entity_embeddings.keys():
                    if entity.lower() in emb_entity.lower() or emb_entity.lower() in entity.lower():
                        entity_embeddings_list.append(self.entity_embeddings[emb_entity])
                        break

        # Mean aggregation (as per DEAP-FAKED paper)
        if entity_embeddings_list:
            combined_embedding = np.mean(entity_embeddings_list, axis=0)
        else:
            combined_embedding = np.zeros(config.kg_embed_dim, dtype=np.float32)

        self.entity_embedding_cache[text] = combined_embedding
        return combined_embedding

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        text = str(row['title'])
        label = int(row['label'])

        sequence = self.text_to_sequence(text)
        kg_embedding = self.get_entity_embedding(text)

        return {
            'sequence': torch.tensor(sequence, dtype=torch.long),
            'kg_embedding': torch.tensor(kg_embedding, dtype=torch.float32),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# ============================================================================
# STEP 5: Build Vocabulary
# ============================================================================

def build_vocabulary(train_df: pd.DataFrame) -> Dict[str, int]:
    """Build vocabulary from training data"""
    print("\nBuilding vocabulary...")

    if os.path.exists(config.vocab_path):
        print("Loading existing vocabulary...")
        with open(config.vocab_path, 'rb') as f:
            return pickle.load(f)

    word_counts = Counter()

    for text in train_df['title']:
        words = str(text).lower().split()
        word_counts.update(words)

    most_common = word_counts.most_common(config.vocab_size - 2)

    vocab = {'<PAD>': 0, '<UNK>': 1}
    for idx, (word, _) in enumerate(most_common):
        vocab[word] = idx + 2

    print(f"Vocabulary size: {len(vocab):,}")

    with open(config.vocab_path, 'wb') as f:
        pickle.dump(vocab, f)

    return vocab

In [ ]:
# ============================================================================
# STEP 6: Model Architecture (DEAP-FAKED Paper)
# ============================================================================

class NewsEncoder(nn.Module):
    """2-layer stacked BiLSTM for news title encoding"""
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int, dropout: float):
        super(NewsEncoder, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.bilstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim // 2,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embedded = self.dropout(self.embedding(x))
        lstm_out, (hidden, _) = self.bilstm(embedded)

        hidden_forward = hidden[-2, :, :]
        hidden_backward = hidden[-1, :, :]

        news_features = torch.cat((hidden_forward, hidden_backward), dim=1)
        return news_features

class EntityEncoder(nn.Module):
    """Entity encoder with KG embeddings"""
    def __init__(self, kg_embed_dim: int, hidden_dim: int):
        super(EntityEncoder, self).__init__()
        self.projection = nn.Linear(kg_embed_dim, hidden_dim)

    def forward(self, kg_embeddings: torch.Tensor) -> torch.Tensor:
        return self.projection(kg_embeddings)

class DEAPFAKED(nn.Module):
    """Complete DEAP-FAKED model"""
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int,
                 kg_embed_dim: int, dropout: float):
        super(DEAPFAKED, self).__init__()

        self.news_encoder = NewsEncoder(vocab_size, embedding_dim, hidden_dim, dropout)
        self.entity_encoder = EntityEncoder(kg_embed_dim, hidden_dim)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, sequence: torch.Tensor, kg_embedding: torch.Tensor) -> torch.Tensor:
        news_features = self.news_encoder(sequence)
        entity_features = self.entity_encoder(kg_embedding)
        combined_features = torch.cat([news_features, entity_features], dim=1)
        logits = self.classifier(combined_features)
        return logits

In [ ]:
# ============================================================================
# STEP 7: Training Utilities
# ============================================================================

class DeapTrainer:
    """Trainer for DEAP-FAKED model"""

    def __init__(self, model: nn.Module, config: DeapConfig):
        self.model = model
        self.config = config
        self.device = config.device

        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

        self.model.to(self.device)

    def train_epoch(self, train_loader: DataLoader) -> Tuple[float, float]:
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0

        for batch in tqdm(train_loader, desc="Training", leave=False):
            sequences = batch['sequence'].to(self.device)
            kg_embeddings = batch['kg_embedding'].to(self.device)
            labels = batch['label'].to(self.device)

            self.optimizer.zero_grad()
            outputs = self.model(sequences, kg_embeddings)
            loss = self.criterion(outputs, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

            self.optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        avg_loss = total_loss / len(train_loader)
        accuracy = correct / total

        return avg_loss, accuracy

    def evaluate(self, data_loader: DataLoader) -> Tuple[float, float, float, List, List]:
        """Evaluate model"""
        self.model.eval()
        total_loss = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in tqdm(data_loader, desc="Evaluating", leave=False):
                sequences = batch['sequence'].to(self.device)
                kg_embeddings = batch['kg_embedding'].to(self.device)
                labels = batch['label'].to(self.device)

                outputs = self.model(sequences, kg_embeddings)
                loss = self.criterion(outputs, labels)

                total_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_loss = total_loss / len(data_loader)
        accuracy = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro')

        return avg_loss, accuracy, f1, all_preds, all_labels

    def train(self, train_loader: DataLoader, val_loader: DataLoader,
              test_loader: DataLoader) -> Tuple[Dict, Dict]:
        """Full training with early stopping"""
        best_val_f1 = 0
        patience_counter = 0
        best_model_state = None

        history = {
            'train_loss': [], 'val_loss': [],
            'train_acc': [], 'val_acc': [],
            'val_f1': []
        }

        print(f"\nStarting training on {self.device}")
        print(f"Model parameters: {sum(p.numel() for p in self.model.parameters()):,}")

        for epoch in range(self.config.num_epochs):
            print(f"\nEpoch {epoch + 1}/{self.config.num_epochs}")
            print("-" * 50)

            train_loss, train_acc = self.train_epoch(train_loader)
            val_loss, val_acc, val_f1, _, _ = self.evaluate(val_loader)

            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['train_acc'].append(train_acc)
            history['val_acc'].append(val_acc)
            history['val_f1'].append(val_f1)

            print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
            print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val F1: {val_f1:.4f}")

            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                patience_counter = 0
                best_model_state = self.model.state_dict().copy()
                print(f"Best model with F1: {val_f1:.4f}")
            else:
                patience_counter += 1
                if patience_counter >= self.config.early_stop_patience:
                    print(f"Early stopping after {epoch + 1} epochs")
                    break

        if best_model_state is not None:
            self.model.load_state_dict(best_model_state)

        test_loss, test_acc, test_f1, test_preds, test_labels = self.evaluate(test_loader)

        print("\n" + "="*60)
        print("DEAP-FAKED WITH DBPEDIA TEST RESULTS")
        print("="*60)
        print(f"Test Loss:    {test_loss:.4f}")
        print(f"Test Accuracy: {test_acc:.4f}")
        print(f"Test F1-Score: {test_f1:.4f}")

        print("\nClassification Report:")
        print(classification_report(test_labels, test_preds,
                                    target_names=['Real', 'Fake'], digits=4))

        results = {
            'test_loss': test_loss,
            'test_accuracy': test_acc,
            'test_f1': test_f1,
            'predictions': test_preds,
            'labels': test_labels,
            'best_val_f1': best_val_f1
        }

        return history, results

In [ ]:
# ============================================================================
# STEP 8: Main Execution Function
# ============================================================================

def run_deap_faked_with_dbpedia():
    """Main function to run DEAP-FAKED implementation with DBpedia"""
    print("="*80)
    print("DEAP-FAKED Implementation with DBpedia as Knowledge Graph")
    print("="*80)
    print("Note: Using DBpedia instead of Wikidata5M (original paper)")
    print("-" * 80)

    # Step 1: Setup
    DataLoaderManager.setup_directories()

    # Step 2: Load preprocessed data
    print("\n1. LOADING PREPROCESSED DATA")
    print("-" * 40)

    train_df, val_df, test_df = DataLoaderManager.load_preprocessed_data()
    if train_df is None:
        return None

    # Step 3: Extract entity names
    print("\n2. EXTRACTING ENTITY NAMES")
    print("-" * 40)

    entity_names = DataLoaderManager.extract_entity_names()
    if not entity_names:
        print("No entity names found, using empty entity set")
        entity_names = []

    # Step 4: Create DBpedia entity embeddings
    print("\n3. CREATING DBPEDIA ENTITY EMBEDDINGS")
    print("-" * 40)

    entity_embeddings = DBpediaLinker.create_dbpedia_embeddings_from_entities(entity_names)
    if not entity_embeddings:
        print("Warning: No entity embeddings created")
        entity_embeddings = {}

    print(f"Entity embedding dimension: {config.kg_embed_dim}")
    print(f"Entity coverage: {len(entity_embeddings):,} embeddings created")

    # Step 5: Build vocabulary
    print("\n4. BUILDING VOCABULARY")
    print("-" * 40)

    vocab = build_vocabulary(train_df)

    # Step 6: Initialize entity processor
    print("\n5. INITIALIZING ENTITY PROCESSOR")
    print("-" * 40)

    entity_processor = EntityProcessor()

    # Step 7: Create datasets
    print("\n6. CREATING DATASETS")
    print("-" * 40)

    train_dataset = DeapFakedDataset(train_df, vocab, entity_processor, entity_embeddings)
    val_dataset = DeapFakedDataset(val_df, vocab, entity_processor, entity_embeddings)
    test_dataset = DeapFakedDataset(test_df, vocab, entity_processor, entity_embeddings)

    sample = train_dataset[0]
    print(f"Sample data shapes:")
    print(f"  Sequence: {sample['sequence'].shape}")
    print(f"  KG Embedding: {sample['kg_embedding'].shape}")

    # Step 8: Create dataloaders
    print("\n7. CREATING DATALOADERS")
    print("-" * 40)

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

    print(f"Train batches: {len(train_loader)}")
    print(f"Validation batches: {len(val_loader)}")
    print(f"Test batches: {len(test_loader)}")

    # Step 9: Create model
    print("\n8. CREATING DEAP-FAKED MODEL")
    print("-" * 40)

    model = DEAPFAKED(
        vocab_size=len(vocab),
        embedding_dim=config.embedding_dim,
        hidden_dim=config.hidden_dim,
        kg_embed_dim=config.kg_embed_dim,
        dropout=config.dropout
    )

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model Architecture:")
    print(f"  Vocabulary size: {len(vocab):,}")
    print(f"  Embedding dimension: {config.embedding_dim}")
    print(f"  Hidden dimension: {config.hidden_dim}")
    print(f"  KG embedding dimension: {config.kg_embed_dim}")
    print(f"  Total parameters: {total_params:,}")

    # Step 10: Train model
    print("\n9. TRAINING MODEL")
    print("-" * 40)

    trainer = DeapTrainer(model, config)
    history, results = trainer.train(train_loader, val_loader, test_loader)

    # Step 11: Save results
    print("\n10. SAVING RESULTS")
    print("-" * 40)

    # Save model
    model_save_path = f"{config.results_dir}/deap_faked_dbpedia_model.pt"
    torch.save({
        'epoch': len(history['train_loss']),
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': trainer.optimizer.state_dict(),
        'vocab': vocab,
        'config': config.__dict__,
        'history': history,
        'results': results
    }, model_save_path)

    print(f"Model saved to: {model_save_path}")

    # Save results to CSV
    results_df = pd.DataFrame([{
        'model': 'DEAP-FAKED (DBpedia)',
        'test_accuracy': results['test_accuracy'],
        'test_f1_score': results['test_f1'],
        'test_loss': results['test_loss'],
        'best_val_f1': results['best_val_f1'],
        'vocab_size': len(vocab),
        'embedding_dim': config.embedding_dim,
        'hidden_dim': config.hidden_dim,
        'kg_embed_dim': config.kg_embed_dim,
        'batch_size': config.batch_size,
        'learning_rate': config.learning_rate,
        'dropout': config.dropout,
        'train_samples': len(train_df),
        'val_samples': len(val_df),
        'test_samples': len(test_df),
        'entity_coverage': len(entity_embeddings),
        'retention_rate': '58.35%',
        'kg_source': 'DBpedia'
    }])

    results_csv_path = f"{config.results_dir}/deap_faked_dbpedia_results.csv"
    results_df.to_csv(results_csv_path, index=False)
    print(f"Results saved to: {results_csv_path}")

    # Save predictions
    predictions_df = pd.DataFrame({
        'true_label': results['labels'],
        'predicted_label': results['predictions']
    })
    predictions_csv_path = f"{config.results_dir}/deap_faked_dbpedia_predictions.csv"
    predictions_df.to_csv(predictions_csv_path, index=False)
    print(f"Predictions saved to: {predictions_csv_path}")

    # Step 12: Print final summary
    print("\n" + "="*80)
    print("DEAP-FAKED WITH DBPEDIA IMPLEMENTATION COMPLETE")
    print("="*80)

    print(f"\nMODEL PERFORMANCE:")
    print(f"  Test Accuracy: {results['test_accuracy']:.4f}")
    print(f"  Test F1-Score: {results['test_f1']:.4f}")
    print(f"  Test Loss:     {results['test_loss']:.4f}")

    print(f"\nDATASET STATISTICS:")
    print(f"  Training samples:   {len(train_df):,}")
    print(f"  Validation samples: {len(val_df):,}")
    print(f"  Test samples:       {len(test_df):,}")
    print(f"  Retention rate:     58.35%")

    print(f"\nKNOWLEDGE GRAPH USED: DBpedia (Alternative to Wikidata5M)")
    print(f"  Entity embeddings created: {len(entity_embeddings):,}")
    print(f"  Embedding dimension: {config.kg_embed_dim}")

    print(f"\nCOMPARISON WITH ORIGINAL DEAP-FAKED PAPER:")
    paper_f1 = 0.89  # DEAP-FAKED paper result
    our_f1 = results['test_f1']

    print(f"  DEAP-FAKED paper (Wikidata5M): F1 = {paper_f1:.3f}")
    print(f"  Our implementation (DBpedia):   F1 = {our_f1:.3f}")
    print(f"  Difference: {our_f1 - paper_f1:+.3f}")

    print(f"\nFILES CREATED:")
    print(f"  Model: {model_save_path}")
    print(f"  Results: {results_csv_path}")
    print(f"  Predictions: {predictions_csv_path}")
    print(f"  Entity embeddings: {config.dbpedia_embeddings_path}")

    print(f"\nREADY FOR COMPARISON WITH KNOW-NET!")

    return results

In [ ]:
# ============================================================================
# STEP 9: Execute the Implementation
# ============================================================================

def verify_files():
    """Verify that all required files are present"""
    print("Verifying required files in root directory...")

    required_files = [
        "train_deap.csv",
        "val_deap.csv",
        "test_deap.csv",
        "entity_linking_stats_deap.csv"
    ]

    all_present = True
    for file in required_files:
        if os.path.exists(file):
            print(f"Found: {file}")
        else:
            print(f"Missing: {file}")
            all_present = False

    if all_present:
        print("\nAll required files found!")
        return True
    else:
        print("\nMissing files. Please upload all files to root directory.")
        return False

In [ ]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("DEAP-FAKED Implementation with DBpedia")
    print("Using DBpedia as Knowledge Graph (Alternative to Wikidata5M)")
    print("="*80)

    if not verify_files():
        print("\nPlease upload these files to the root directory:")
        print("  1. train_deap.csv")
        print("  2. val_deap.csv")
        print("  3. test_deap.csv")
        print("  4. entity_linking_stats_deap.csv")
        print("\nThen run the code again.")
    else:
        print("\n" + "="*80)
        print("STARTING DEAP-FAKED WITH DBPEDIA IMPLEMENTATION")
        print("="*80)

        try:
            results = run_deap_faked_with_dbpedia()
            if results:
                print("\n" + "="*80)
                print("DEAP-FAKED WITH DBPEDIA COMPLETED SUCCESSFULLY!")
                print("="*80)
                print(f"\nFinal Results:")
                print(f"   Test Accuracy: {results['test_accuracy']:.4f}")
                print(f"   Test F1-Score: {results['test_f1']:.4f}")
                print(f"\nAll results saved to: {config.results_dir}/")
        except Exception as e:
            print(f"\nError during execution: {e}")
            import traceback
            traceback.print_exc()

DEAP-FAKED Implementation with DBpedia
Using DBpedia as Knowledge Graph (Alternative to Wikidata5M)
Verifying required files in root directory...
Found: train_deap.csv
Found: val_deap.csv
Found: test_deap.csv
Found: entity_linking_stats_deap.csv

All required files found!

STARTING DEAP-FAKED WITH DBPEDIA IMPLEMENTATION
DEAP-FAKED Implementation with DBpedia as Knowledge Graph
Note: Using DBpedia instead of Wikidata5M (original paper)
--------------------------------------------------------------------------------
Results will be saved to: deap_faked_dbpedia_results

1. LOADING PREPROCESSED DATA
----------------------------------------
Loading preprocessed data...
Train samples: 20,953
Validation samples: 2,619
Test samples: 2,620

Class Distribution:
  Train: Real=10,311 (49.2%), Fake=10,642 (50.8%)
  Validation: Real=1,289 (49.2%), Fake=1,330 (50.8%)
  Test: Real=1,289 (49.2%), Fake=1,331 (50.8%)

2. EXTRACTING ENTITY NAMES
----------------------------------------

Extracting entity 

Processing entity stats: 44888it [00:03, 14372.93it/s]


Extracted 2,786 unique entity names

3. CREATING DBPEDIA ENTITY EMBEDDINGS
----------------------------------------

Creating DBpedia entity embeddings...
Error using GloVe: File is not a zip file
Creating simple deterministic embeddings...
Saved 2,786 DBpedia entity embeddings
Entity embedding dimension: 300
Entity coverage: 2,786 embeddings created

4. BUILDING VOCABULARY
----------------------------------------

Building vocabulary...
Vocabulary size: 10,000

5. INITIALIZING ENTITY PROCESSOR
----------------------------------------

Initializing spaCy NER model...
spaCy model loaded

6. CREATING DATASETS
----------------------------------------
Sample data shapes:
  Sequence: torch.Size([256])
  KG Embedding: torch.Size([300])

7. CREATING DATALOADERS
----------------------------------------
Train batches: 655
Validation batches: 82
Test batches: 82

8. CREATING DEAP-FAKED MODEL
----------------------------------------
Model Architecture:
  Vocabulary size: 10,000
  Embedding dimens

Train Loss: 0.2074 | Train Acc: 0.9149
Val Loss:   0.1403 | Val Acc:   0.9530 | Val F1: 0.9530
Best model with F1: 0.9530

Epoch 2/100
--------------------------------------------------


Train Loss: 0.0994 | Train Acc: 0.9643
Val Loss:   0.1322 | Val Acc:   0.9591 | Val F1: 0.9591
Best model with F1: 0.9591

Epoch 3/100
--------------------------------------------------


Train Loss: 0.0658 | Train Acc: 0.9757
Val Loss:   0.1293 | Val Acc:   0.9611 | Val F1: 0.9611
Best model with F1: 0.9611

Epoch 4/100
--------------------------------------------------


Train Loss: 0.0412 | Train Acc: 0.9849
Val Loss:   0.1409 | Val Acc:   0.9702 | Val F1: 0.9702
Best model with F1: 0.9702

Epoch 5/100
--------------------------------------------------


Train Loss: 0.0314 | Train Acc: 0.9886
Val Loss:   0.1490 | Val Acc:   0.9660 | Val F1: 0.9660

Epoch 6/100
--------------------------------------------------


Train Loss: 0.0263 | Train Acc: 0.9914
Val Loss:   0.1678 | Val Acc:   0.9679 | Val F1: 0.9679
Early stopping after 6 epochs



DEAP-FAKED WITH DBPEDIA TEST RESULTS
Test Loss:    0.1294
Test Accuracy: 0.9714
Test F1-Score: 0.9714

Classification Report:
              precision    recall  f1-score   support

        Real     0.9655    0.9767    0.9711      1289
        Fake     0.9772    0.9662    0.9717      1331

    accuracy                         0.9714      2620
   macro avg     0.9713    0.9715    0.9714      2620
weighted avg     0.9714    0.9714    0.9714      2620


10. SAVING RESULTS
----------------------------------------
Model saved to: deap_faked_dbpedia_results/deap_faked_dbpedia_model.pt
Results saved to: deap_faked_dbpedia_results/deap_faked_dbpedia_results.csv
Predictions saved to: deap_faked_dbpedia_results/deap_faked_dbpedia_predictions.csv

DEAP-FAKED WITH DBPEDIA IMPLEMENTATION COMPLETE

MODEL PERFORMANCE:
  Test Accuracy: 0.9714
  Test F1-Score: 0.9714
  Test Loss:     0.1294

DATASET STATISTICS:
  Training samples:   20,953
  Validation samples: 2,619
  Test samples:       2,620
  Reten